## Profiling the candidate's list

In [2]:
import os

print(os.getcwd())
print(os.listdir())

c:\Users\YaaAntwi\Documents\StreamVault\explore
['profile_ids.ipynb']


In [3]:
## Importing the required libraries

from pathlib import Path
import sys

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
from pipeline.config import CANDIDATE_LIST

In [4]:
## Loading the data 
movie_ids = pd.read_csv(
    CANDIDATE_LIST,
    dtype=str,
    keep_default_na=False,
    skip_blank_lines=False
)

In [5]:
print(movie_ids.head())
print(movie_ids.shape)
print(movie_ids.dtypes)

  tmdb_id
0     155
1     165
2     242
3   49026
4    8077
(127, 1)
tmdb_id    str
dtype: object


### Profile the column

In [ ]:
# Missing IDs
missing_ids = (movie_ids["tmdb_id"] == "").sum()

# Distinct IDs
unique_ids = movie_ids["tmdb_id"].nunique()


# Duplicated IDs

duplicate_ids = movie_ids["tmdb_id"].duplicated().sum()

print("Duplicate entries:", duplicate_ids)
print("Duplicate IDs:")
print(movie_ids[movie_ids["tmdb_id"].duplicated(keep=False)])  # displays every row belonging to a duplicate group.
# Most common IDs
most_common = movie_ids["tmdb_id"].value_counts().head(10)

print("Total rows:", len(movie_ids)) ## Total rows
print("Missing_ids", missing_ids)
print("Distinct_IDs", unique_ids)

print("Most common IDs:", most_common)

Duplicate entries: 6
Duplicate IDs:
    tmdb_id
0       155
6       680
25      278
38      603
41      603
53     1891
58      278
78      550
88     1891
89      155
92      550
100     680
Total rows: 127
Missing_ids 1
Distinct_IDs 121
Most common IDs: tmdb_id
155      2
680      2
278      2
603      2
1891     2
550      2
165      1
242      1
49026    1
8077     1
Name: count, dtype: int64


In [9]:
suspicious_ids = movie_ids[
    ~movie_ids["tmdb_id"].str.isdigit()
]

print("Suspicious IDs:")
print(suspicious_ids)

Suspicious IDs:
       tmdb_id
28          -1
67            
93   tt0468569
109        N/A


In [10]:
print("Numeric IDs:", movie_ids["tmdb_id"].str.isdigit().sum())

print("Non-numeric entries:", (~movie_ids["tmdb_id"].str.isdigit()).sum())

print("Unique numeric IDs:", movie_ids[
    movie_ids["tmdb_id"].str.isdigit()
]["tmdb_id"].nunique())

Numeric IDs: 123
Non-numeric entries: 4
Unique numeric IDs: 117


In [11]:
suspicious_ids = movie_ids[
    ~movie_ids["tmdb_id"].str.isdigit()
].copy()

suspicious_ids["issue"] = "Non-numeric ID"

suspicious_ids.loc[
    suspicious_ids["tmdb_id"] == "tt0468569", "issue"
] = "IMDb ID"

suspicious_ids.loc[
    suspicious_ids["tmdb_id"] == "-1", "issue"
] = "Invalid-looking ID"

suspicious_ids.loc[
    suspicious_ids["tmdb_id"] == "", "issue"
] = "Blank ID"

suspicious_ids.loc[
    suspicious_ids["tmdb_id"] == "N/A", "issue"
] = "Missing-value marker"

print(suspicious_ids)

       tmdb_id                 issue
28          -1    Invalid-looking ID
67                          Blank ID
93   tt0468569               IMDb ID
109        N/A  Missing-value marker


In [12]:
print("\n CANDIDATE LIST SUMMARY ")

print("Original entries:", len(movie_ids))

print("Blank IDs:", (movie_ids["tmdb_id"] == "").sum())

print("N/A entries:", (movie_ids["tmdb_id"] == "N/A").sum())

print("Invalid-looking -1 entries:", (movie_ids["tmdb_id"] == "-1").sum())

print("IMDb IDs:", movie_ids["tmdb_id"].str.startswith("tt").sum())

print("Unique numeric IDs:", movie_ids[
    movie_ids["tmdb_id"].str.isdigit()
]["tmdb_id"].nunique())


 CANDIDATE LIST SUMMARY 
Original entries: 127
Blank IDs: 1
N/A entries: 1
Invalid-looking -1 entries: 1
IMDb IDs: 1
Unique numeric IDs: 117
